In [1]:
import helpers.data as data
import helpers.features as features
import joblib
import hmmlearn.hmm as hmm
import numpy as np  
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from sklearn.ensemble import GradientBoostingRegressor

In [2]:
hmm_features = ["mean_spread", "spread_std", "vol_accel", "har_rv"]
# Use only hmm_state for live predictions (no data leakage)
# Previous version used ["hmm_state", "target"] which caused data leakage
gbt_features = ["hmm_state", "mean_divergence", "har_sigma"]
train_size = 0.6
sequence_length = 24


In [3]:
df = data.get_data()
df = features.add_features(df)
df = df.dropna()

print(df.columns)

Features computed in 1.687s
Index(['time', 'open', 'high', 'low', 'close', 'volume', 'new_session',
       'har_rv', 'session_vwap', 'ema', 'val', 'vah', 'poc', 'vol_accel',
       'mean_spread', 'spread_std', 'composite_mean', 'mean_divergence',
       'har_sigma'],
      dtype='object')


Train HMM

In [4]:
# hmm = hmm.GaussianHMM(n_components=3, covariance_type="diag")
# hmm.fit(df[hmm_features].iloc[:int(len(df) * train_size)])
# joblib.dump(hmm, "./trained_models/regime_hmm.pkl")

Load HMM

In [5]:
hmm = joblib.load("./trained_models/regime_hmm.pkl")
df["hmm_state"] = hmm.predict(df[hmm_features])

Train GBT

In [6]:
df = features.add_target(df)
# Use only hmm_state features (no target in features to avoid data leakage)
train_df = df[gbt_features].iloc[:int(len(df) * train_size)]
targets = df['target'].iloc[:int(len(df) * train_size)].values
print(train_df)
print(f"\nTraining with {len(gbt_features)} feature(s): {gbt_features}")
print(f"Input shape will be: {sequence_length} candles × {len(gbt_features)} features = {sequence_length * len(gbt_features)} features")

def create_sequences(data, target_values):
    target_values = target_values.clip(-3,3)
    feature_data = data[gbt_features].values
    
    X_sequences = []
    y_targets = []
    
    # Create sliding windows
    for i in range(sequence_length, len(data)):
        # Get sequence of previous candles (not including current candle)
        sequence = feature_data[i - sequence_length:i]
        # Flatten the sequence into a single feature vector
        flattened_sequence = sequence.flatten()
        X_sequences.append(flattened_sequence)
        # Target is the current candle's target value (what we're predicting)
        y_targets.append(target_values[i])
    
    return np.array(X_sequences), np.array(y_targets)

X, Y = create_sequences(train_df, targets)
print(f"\nTraining data shape: X={X.shape}, Y={Y.shape}")

model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=40,
    verbose=1
)
# Fit with sample weights to emphasize chop regime samples
model.fit(X, Y)

joblib.dump(model, "./trained_models/gbt.pkl")

                           hmm_state  mean_divergence  har_sigma
timestamp                                                       
2025-03-13 23:10:00+00:00          2      1554.235566   8.611003
2025-03-13 23:15:00+00:00          2      -912.515907   8.543708
2025-03-13 23:20:00+00:00          2       958.720412   8.540060
2025-03-13 23:25:00+00:00          2      -321.539994   8.521990
2025-03-13 23:30:00+00:00          2      1524.087404   8.508376
...                              ...              ...        ...
2025-10-13 15:00:00+00:00          2     -5575.528497  11.915153
2025-10-13 15:05:00+00:00          2      -617.704763  11.912994
2025-10-13 15:10:00+00:00          2      5444.684789  11.903532
2025-10-13 15:15:00+00:00          2     11496.342037  11.869895
2025-10-13 15:20:00+00:00          2     11258.842746  11.881148

[41358 rows x 3 columns]

Training with 3 feature(s): ['hmm_state', 'mean_divergence', 'har_sigma']
Input shape will be: 24 candles × 3 features = 72 feat


Training data shape: X=(41334, 72), Y=(41334,)
      Iter       Train Loss   Remaining Time 
         1           0.0001            1.23m
         2           0.0001            1.22m
         3           0.0001            1.19m
         4           0.0001            1.18m
         5           0.0001            1.16m
         6           0.0001            1.15m
         7           0.0001            1.14m
         8           0.0001            1.12m
         9           0.0001            1.11m
        10           0.0001            1.11m
        20           0.0001           59.71s
        30           0.0001           52.45s
        40           0.0001           44.88s
        50           0.0001           37.42s
        60           0.0001           29.88s
        70           0.0001           22.39s
        80           0.0001           14.95s
        90           0.0001            7.48s
       100           0.0001            0.00s


['./trained_models/gbt.pkl']